# Noisy-AR eval — aggregate & pivot

Walks `runs-noisy-ar/**/eval_noisy/NB*/metrics.json` and builds the comparison table.
Logic lives in `collect_noisy_eval.py`; this notebook is just a thin driver.

In [1]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'eval' else 'eval')
from collect_noisy_eval import collect, derive_write_frequency, pivot
import pandas as pd
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

In [2]:
RUNS_ROOT = '/home/bulatov/rmt/test-time/compressing-associations-gdn/runs-noisy-ar'   # adjust if needed
df = collect(RUNS_ROOT)
df['write_frequency'] = derive_write_frequency(df)
print(f'loaded {len(df)} (run, NB) rows from {df["run_name"].nunique()} runs')
df.head()

loaded 91 (run, NB) rows from 9 runs


,NB,seed,exact_match,token_accuracy,metrics_path,run_name,model,write_frequency,L,H,D,state_size,lr,bs,version,train_dataset,N_pairs,K_train,vary,B,M,pps,tps,read_mode_parsed,write_mode_parsed,write_value_dim,read_mode,write_mode,write_config,config_label
0,128,1,0.9965,0.99825,/home/bulatov/rmt/test-time/compressing-associ...,gdn_gated_delta_net_L4H4D128_ss32_ck4_lr1e-03_...,GDN,1 token,4,4,128,32,0.001,64,gdn,N4-K2V2-V62_K1-vary-B7_1M,4,1,True,7,NaN,NaN,NaN,NaN,NaN,<NA>,N/A,N/A,N/A,gdn|N/A→N/A
1,16,1,0.9960,0.99775,/home/bulatov/rmt/test-time/compressing-associ...,gdn_gated_delta_net_L4H4D128_ss32_ck4_lr1e-03_...,GDN,1 token,4,4,128,32,0.001,64,gdn,N4-K2V2-V62_K1-vary-B7_1M,4,1,True,7,NaN,NaN,NaN,NaN,NaN,<NA>,N/A,N/A,N/A,gdn|N/A→N/A
2,32,1,0.9975,0.99875,/home/bulatov/rmt/test-time/compressing-associ...,gdn_gated_delta_net_L4H4D128_ss32_ck4_lr1e-03_...,GDN,1 token,4,4,128,32,0.001,64,gdn,N4-K2V2-V62_K1-vary-B7_1M,4,1,True,7,NaN,NaN,NaN,NaN,NaN,<NA>,N/A,N/A,N/A,gdn|N/A→N/A
3,4,1,0.9985,0.99925,/home/bulatov/rmt/test-time/compressing-associ...,gdn_gated_delta_net_L4H4D128_ss32_ck4_lr1e-03_...,GDN,1 token,4,4,128,32,0.001,64,gdn,N4-K2V2-V62_K1-vary-B7_1M,4,1,True,7,NaN,NaN,NaN,NaN,NaN,<NA>,N/A,N/A,N/A,gdn|N/A→N/A
4,64,1,0.9960,0.99800,/home/bulatov/rmt/test-time/compressing-associ...,gdn_gated_delta_net_L4H4D128_ss32_ck4_lr1e-03_...,GDN,1 token,4,4,128,32,0.001,64,gdn,N4-K2V2-V62_K1-vary-B7_1M,4,1,True,7,NaN,NaN,NaN,NaN,NaN,<NA>,N/A,N/A,N/A,gdn|N/A→N/A


In [ ]:
def get_write_config(model_name):

In [4]:
index_cols=("N_pairs", "model", "write_frequency", "M", "lr",
            "read_mode_parsed",
            "write_mode_parsed",
            # "pps",
            # 'seed',
            )
value_col="exact_match"
nb_order=(4, 8, 16, 32, 64, 128)
agg="mean"

grouped = (
    df.groupby(list(index_cols) + ["NB"], dropna=False)[value_col]
        .agg(agg)
)
grouped

N_pairs  model  write_frequency  M    lr      read_mode_parsed  write_mode_parsed  NB 
4        GDN    1 token          NaN  0.0010  NaN               NaN                4      0.99850
                                                                                   8      0.99950
                                                                                   16     0.99600
                                                                                   32     0.99750
                                                                                   64     0.99600
                                                                                   128    0.99650
         RMM    1 pair           4.0  0.0001  unpool            pool               4      0.56825
                                                                                   8      0.54275
                                                                                   16     0.82250
                               

In [8]:
# Mean exact-match across seeds, pivot to wide.
wide = pivot(df, value_col='exact_match', agg='mean')
wide.round(4)

,,,,,NB4,NB8,NB16,NB32,NB64,NB128
model,write_frequency,M,lr,N_pairs,,,,,,
GDN,1 token,NaN,0.0010,4,0.9985,0.9995,0.9960,0.9975,0.9960,0.9965
RMM,1 pair,4.0,0.0001,4,0.5682,0.5428,0.8225,0.7825,0.6455,NaN


In [10]:
# Std across seeds, side by side.
wide_mean = pivot(df, value_col='exact_match', agg='mean').round(4)
wide_std  = pivot(df, value_col='exact_match', agg='std').round(4)
combined = wide_mean.astype(str) + ' ± ' + wide_std.astype(str)
combined

,,,,,NB4,NB8,NB16,NB32,NB64,NB128
model,write_frequency,M,lr,N_pairs,,,,,,
GDN,1 token,NaN,0.0010,4,NaN,NaN,NaN,NaN,NaN,NaN
RMM,1 pair,4.0,0.0001,4,0.5682 ± 0.4317,0.5428 ± 0.431,NaN,NaN,NaN,NaN


In [ ]:
# # Optional: save
# wide_mean.to_csv('eval/noisy_eval_table.csv')
# df.to_csv('eval/noisy_eval_long.csv', index=False)
# print('saved → eval/noisy_eval_table.csv, eval/noisy_eval_long.csv')